# Module 6 · Embeddings & Semantic Search
Turn text into vectors. Measure semantic similarity. Build a semantic search engine.

---
> **Setup:** Run the first two cells before anything else.
> API key is loaded automatically from the `.env` file in the parent folder.

In [1]:
%pip install -q google-genai python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, json, math, time, base64, getpass, re, urllib.request
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv()  # picks up modules/.env symlink
api_key = os.environ.get('GEMINI_API_KEY')
if not api_key:
    api_key = getpass.getpass('Paste your Gemini API key: ')

client = genai.Client(api_key=api_key)
MODEL = 'gemma-4-31b-it'
DEFAULT_MAX = 2048

def cfg(**kwargs):
    kwargs.setdefault('max_output_tokens', DEFAULT_MAX)
    kwargs.setdefault('temperature', 0.7)
    return types.GenerateContentConfig(**kwargs)

def get_text(response) -> str:
    if response.text:
        return response.text.strip()
    parts = []
    for cand in (response.candidates or []):
        if cand.content:
            for part in cand.content.parts:
                if not getattr(part, 'thought', False) and part.text:
                    parts.append(part.text)
    return ''.join(parts).strip()

def _call_with_retry(fn, *args, max_retries=5, **kwargs):
    for attempt in range(max_retries):
        try:
            return fn(*args, **kwargs)
        except Exception as e:
            msg = str(e)
            if '429' in msg or 'RESOURCE_EXHAUSTED' in msg:
                # parse suggested retry delay from error (e.g. 'retryDelay': '30s')
                m = re.search(r'retry[^0-9]*([0-9]+)s', msg, re.I)
                wait = int(m.group(1)) + 5 if m else 35
                print(f'  ⏳ Rate-limited — waiting {wait}s (attempt {attempt+1}/{max_retries})')
                time.sleep(wait)
            else:
                raise
    raise RuntimeError('Max retries exceeded')

_raw_gen    = client.models.generate_content
_raw_stream = client.models.generate_content_stream
_raw_embed  = client.models.embed_content
_raw_count  = client.models.count_tokens
client.models.generate_content        = lambda *a,**kw: _call_with_retry(_raw_gen,    *a,**kw)
client.models.generate_content_stream = lambda *a,**kw: _call_with_retry(_raw_stream, *a,**kw)
client.models.embed_content           = lambda *a,**kw: _call_with_retry(_raw_embed,  *a,**kw)
client.models.count_tokens            = lambda *a,**kw: _call_with_retry(_raw_count,  *a,**kw)

print('\u2705 Setup complete. Model:', MODEL, '| Retry-on-429: enabled')


✅ Setup complete. Model: gemma-4-31b-it | Retry-on-429: enabled


### 🔌 API Key Test

In [3]:
try:
    _r = client.models.generate_content(
        model=MODEL,
        contents="Reply with exactly the words: Hello LLM course",
        config=cfg(temperature=0.0)
    )
    print("✅ API key working!")
    print("Model says:", get_text(_r))
except Exception as e:
    print("❌ API key error:", e)

✅ API key working!
Model says: Hello LLM course


---
## 10. Embeddings & Semantic Similarity

LLMs can also **represent text as numbers** — this is called an **embedding**.

An embedding is a list of ~3072 numbers (a vector) that captures the *meaning* of a piece of text.

```
"king"   → [0.23, -0.71, 0.45, ...  3072 numbers]
"queen"  → [0.21, -0.69, 0.43, ...  3072 numbers]  ← very similar!
"banana" → [-0.54, 0.12, -0.88, ... 3072 numbers]  ← very different
```

**Cosine similarity:** 1.0 = identical meaning, 0.0 = unrelated, −1.0 = opposite.

**Use cases:** semantic search, clustering, recommendation, RAG (retrieval-augmented generation).

In [4]:
EMBED_MODEL = "gemini-embedding-001"

def embed(text: str) -> list:
    result = client.models.embed_content(model=EMBED_MODEL, contents=text)
    return result.embeddings[0].values

def cosine_similarity(a: list, b: list) -> float:
    dot = sum(x * y for x, y in zip(a, b))
    mag_a = math.sqrt(sum(x**2 for x in a))
    mag_b = math.sqrt(sum(x**2 for x in b))
    return dot / (mag_a * mag_b)

sentences = [
    "I love programming in Python.",
    "Python is my favourite coding language.",
    "The weather today is sunny and warm.",
    "Machine learning uses data to train models.",
    "Deep learning is a subset of machine learning.",
]

embeddings = {s: embed(s) for s in sentences}
print(f"Embedding dimension: {len(list(embeddings.values())[0])}")
print(f"First 5 values: {list(embeddings.values())[0][:5]}")

Embedding dimension: 3072
First 5 values: [-0.007137808, -0.00094349345, 0.0070928317, -0.0890881, -0.009671789]


In [5]:
# Similarity matrix — high values = similar meaning
labels = [s[:38] + "..." if len(s) > 38 else s for s in sentences]
max_len = max(len(l) for l in labels)

print("Cosine similarity (1.0 = identical meaning):\n")
header = " " * (max_len + 2) + "  ".join(f"[{i+1}]" for i in range(len(sentences)))
print(header)
for i, s1 in enumerate(sentences):
    row = f"[{i+1}] {labels[i]:<{max_len}}  "
    for s2 in sentences:
        sim = cosine_similarity(embeddings[s1], embeddings[s2])
        row += f"{sim:.2f}  "
    print(row)

Cosine similarity (1.0 = identical meaning):

                                           [1]  [2]  [3]  [4]  [5]
[1] I love programming in Python.              1.00  0.84  0.52  0.54  0.53  
[2] Python is my favourite coding language...  0.84  1.00  0.55  0.55  0.54  
[3] The weather today is sunny and warm.       0.52  0.55  1.00  0.54  0.51  
[4] Machine learning uses data to train mo...  0.54  0.55  0.54  1.00  0.72  
[5] Deep learning is a subset of machine l...  0.53  0.54  0.51  0.72  1.00  


In [6]:
# Semantic search: find the most similar sentence to a query
def semantic_search(query: str, corpus: list) -> list:
    q_vec = embed(query)
    scores = [(cosine_similarity(q_vec, embed(doc)), doc) for doc in corpus]
    return sorted(scores, reverse=True)

query = "What coding language should I learn?"
results = semantic_search(query, sentences)

print(f"Query: '{query}'\n")
for score, doc in results:
    print(f"  {score:.3f}  {doc}")

Query: 'What coding language should I learn?'

  0.711  Python is my favourite coding language.
  0.598  I love programming in Python.
  0.555  Machine learning uses data to train models.
  0.531  Deep learning is a subset of machine learning.
  0.509  The weather today is sunny and warm.
